# Sailor-7B-Chat Task 1 → Task 2 locally

Run this notebook from the repository root or `notebooks/` on Linux/WSL. It uses the existing checkout and training interpreter: **no clone, package installation, environment creation, or kernel restart**. Install the repository training dependencies in your local CUDA-compatible environment before running; select it with `TRAINING_PYTHON`, `GECC_TRAINING_PYTHON`, an activated environment, or a discovered `.venv`/`venv`. The notebook kernel needs IPython; heavy training imports run only in the training interpreter.

Default training is **one CUDA GPU** with the canonical selected recipes unchanged: Task 1 has five epochs, LR 5e-5, effective batch eight, seed 42, LoRA rank 16; five leakage-safe OOF fits have three epochs each. Task 2 has eight epochs and LR 2e-5. Moving from two GPUs to one changes accumulation automatically, not the effective batch. Task 1 selects validation macro F1; Task 2 selects error-only GLEU+, then exact match, then clean preservation. These are independent adapters; classifier weights are not transferred into the corrector, and predicted class 0 never bypasses correction.

Set explicit local data paths, or use automatic unambiguous lookup in `DATA_ROOT` (default `dataset/`): `VGEC 2026_v1.json`, `VGEC_classifier.json`, `VGEC_balanced.json`. Fresh Task 1 with OOF needs all three training files. `SUBMISSION_DATA="auto"` separately discovers `public_test.json` under `DATA_ROOT` or repository `dataset/` for inference only. Output goes to repository `runs/`, with portable handoffs, configs, streamed logs and reports. After Task 2, saved adapters infer the submission input into `RUN/submission.json`; no public-test scores are fabricated because labels are absent. Missing automatic input (or `SUBMISSION_DATA=None`) exports the labeled validation set after training. Keep corpus-derived data and text-containing outputs private.

W&B tracking is enabled by default. Provide `WANDB_API_KEY` through the environment; never paste a key into a notebook cell. `WANDB_MODE=offline` supports deferred sync; `TRACKING_ENABLED=False` disables tracking. Main fits, OOF fits and diagnostics share one group. Model artifacts and non-text reports are logged by default; corpus text/hints cannot be uploaded under the supplied agreement. Keep projects private; private hosting does not itself grant redistribution rights.

CPU is an **explicit experiment**, not an automatic fallback. Set `TRAINING_DEVICE="cpu"` and supply CPU-compatible configuration overrides (at least `model.load_in_4bit=False` for both stages and classifier `training.args.optim="adamw_torch"`). The full unquantized 7B recipe needs substantial host memory; this notebook does not silently replace it with a smaller model. `DIAGNOSTICS_DEVICE` also selects saved-run inference placement; a saved NF4 adapter requires CUDA.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import signal
import subprocess
import sys

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")

PROJECT_PATH = None  # Optional checkout path; otherwise discover from cwd and its parents.
TRAINING_PYTHON = None  # Existing interpreter path; never installs or mutates packages.
DATA_ROOT = None  # Defaults to PROJECT_DIR / "dataset"; relative paths are repo-relative.
OUTPUT_PATH = "runs"  # Relative to PROJECT_DIR, or an absolute local directory.
CLASSIFICATION_CONFIG = "configs/baseline.json"
CORRECTION_CONFIG = "configs/correction.json"
CLASSIFICATION_OVERRIDES = {}  # Explicit experiments only; default recipe stays unchanged.
CORRECTION_OVERRIDES = {}
TRAINING_DEVICE = "cuda"  # "cpu" requires explicit CPU-compatible model/optimizer overrides.
WORLD_SIZE = 1
DIAGNOSTICS_DEVICE = "cuda"  # Independent of the training device.
RUN_MODE = "both"  # task1, task2 (requires prior classifier), or both.
RUN_TRAINING = True  # False runs saved-adapter inference and/or enabled diagnostics.
GENERATE_OOF = True
PRIOR_RUN = None  # e.g. "runs/sailor-7b-both-..."; extracted completed run, not a ZIP.
CLASSIFIER_RUN = None  # Optional direct classifier run; takes precedence over PRIOR_RUN.
RESUME_CORRECTION = False  # Warmstart prior correction weights with a NEW optimizer/scheduler.
INITIAL_ADAPTER = None  # Optional explicit compatible corrector, instead of RESUME_CORRECTION.
EDIT_CORRECTION = False  # Fresh adapter: localized character edits, not full-sentence targets.
HINT_ABLATION = False
CANDIDATE_RERANKING = False
EXPERIMENT_RUN = None  # This output run by default, or PRIOR_RUN when RUN_TRAINING=False.
ORIGINAL_DATA = None  # Explicit absolute or repo-relative private JSON path.
CLASSIFIER_DATA = None
CORRECTION_DATA = None
SUBMISSION_DATA = "auto"  # Discover public_test.json for inference; explicit path overrides; None uses validation.
TRACKING_ENABLED = True
WANDB_PROJECT = "vlsp26-gec"
WANDB_ENTITY = None
WANDB_MODE = os.environ.get("WANDB_MODE", "online")  # online, offline, or disabled.
WANDB_LOG_MODELS = True
WANDB_LOG_TEXT = False  # Must stay false: the corpus agreement prohibits redistribution.
WANDB_GROUP = os.environ.get("WANDB_RUN_GROUP")  # None groups jobs by this new output run.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")


In [ ]:
required_files = ("model/two_phase.py", "model/correction_experiments.py",
                  "configs/baseline.json", "configs/correction.json")
if PROJECT_PATH:
    PROJECT_DIR = Path(PROJECT_PATH).expanduser().resolve()
else:
    here = Path.cwd().resolve()
    PROJECT_DIR = next((root for root in (here, *here.parents)
                        if all((root / name).is_file() for name in required_files)), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError("Run from the repository or notebooks directory, or set PROJECT_PATH to the checkout.")
for name in required_files:
    if not (PROJECT_DIR / name).is_file():
        raise FileNotFoundError(PROJECT_DIR / name)

# All user-supplied relative paths have the same meaning from repo root and notebooks/.
def local_path(value, *, must_exist=False):
    path = Path(value).expanduser()
    path = (path if path.is_absolute() else PROJECT_DIR / path).resolve()
    if must_exist and not path.exists():
        raise FileNotFoundError(path)
    return path

interpreter = TRAINING_PYTHON or os.environ.get("GECC_TRAINING_PYTHON")
if interpreter:
    interpreter = Path(interpreter).expanduser()
    if not interpreter.is_absolute():
        interpreter = PROJECT_DIR / interpreter
else:
    active_env = os.environ.get("VIRTUAL_ENV") or os.environ.get("CONDA_PREFIX")
    environments = ([Path(active_env)] if active_env else []) + [
        root / name for root in (PROJECT_DIR, PROJECT_DIR.parent) for name in (".venv", "venv")]
    interpreter = next((root / "bin/python" for root in environments
                        if (root / "bin/python").is_file()), Path(sys.executable))
# Do not resolve the interpreter symlink: venv Python must retain its environment location.
TRAINING_PYTHON = str(interpreter.absolute())
if not interpreter.is_file() or not os.access(interpreter, os.X_OK):
    raise FileNotFoundError(f"Training interpreter is not executable: {interpreter}")
DATA_ROOT = local_path(DATA_ROOT or "dataset")
OUTPUT_ROOT = local_path(OUTPUT_PATH)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = OUTPUT_ROOT / f"sailor-7b-{RUN_MODE}-{RUN_ID}"

sys.path.insert(0, str(PROJECT_DIR))
from tools.runtime import source_identity
from model.runtime import load_config, resolve_data, resolve_submission_data
from model.run_reuse import resolve_run
SOURCE_INFO = source_identity(PROJECT_DIR)
print("Checkout:", PROJECT_DIR, "| training interpreter:", TRAINING_PYTHON)
print("Input root:", DATA_ROOT, "| output run:", OUTPUT_DIR)
print(json.dumps(SOURCE_INFO, indent=2))


In [ ]:
def apply_overrides(target, overrides):
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(target.get(key), dict):
            apply_overrides(target[key], value)
        else:
            target[key] = value

# Environment-only credentials never enter commands, saved configs or notebook output.
def redact(text):
    for name in ("WANDB_API_KEY", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        secret = os.environ.get(name)
        if secret:
            text = text.replace(secret, "[REDACTED]")
    return text

def stream_command(arguments, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Launching:", redact(" ".join(map(str, arguments))))
    with log_path.open("w", encoding="utf-8") as log_file:
        log_file.write(json.dumps(SOURCE_INFO, ensure_ascii=False) + "\n")
        log_file.flush()
        with subprocess.Popen(arguments, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1,
                              start_new_session=True) as process:
            try:
                for line in process.stdout:
                    line = redact(line)
                    print(line, end="", flush=True)
                    log_file.write(line)
                    log_file.flush()
                returncode = process.wait()
            except BaseException as error:
                # Stop the complete launcher/trainer process group, not just its parent.
                try:
                    os.killpg(process.pid, signal.SIGINT if isinstance(error, KeyboardInterrupt) else signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                    process.wait()
                raise
        if returncode:
            raise subprocess.CalledProcessError(returncode, arguments)


In [ ]:
if RUN_MODE not in ("task1", "task2", "both"):
    raise ValueError("RUN_MODE must be task1, task2, or both")
if TRAINING_DEVICE not in ("cuda", "cpu") or DIAGNOSTICS_DEVICE not in ("cuda", "cpu"):
    raise ValueError("Devices must be cuda or cpu")
if RESUME_CORRECTION and INITIAL_ADAPTER:
    raise ValueError("Choose RESUME_CORRECTION or INITIAL_ADAPTER, not both")
if RESUME_CORRECTION and not PRIOR_RUN:
    raise ValueError("RESUME_CORRECTION requires PRIOR_RUN")
if not (RUN_TRAINING and RUN_MODE == "task1" and SUBMISSION_DATA == "auto"):
    submission_path = resolve_submission_data(
        SUBMISSION_DATA if SUBMISSION_DATA in (None, "auto") else local_path(SUBMISSION_DATA),
        input_root=DATA_ROOT)
    SUBMISSION_DATA = str(submission_path) if submission_path is not None else None
else:
    SUBMISSION_DATA = None
print("Submission input:", SUBMISSION_DATA or "validation after training (no public test selected)")
if not RUN_TRAINING and not (SUBMISSION_DATA or HINT_ABLATION or CANDIDATE_RERANKING):
    raise ValueError("With RUN_TRAINING=False, set SUBMISSION_DATA or enable a correction diagnostic")
if RUN_TRAINING and RUN_MODE == "task1" and (HINT_ABLATION or CANDIDATE_RERANKING):
    raise ValueError("Task 1 has no corrector; select both/task2 or saved-run experiments")

if RUN_TRAINING and RUN_MODE == "task1" and SUBMISSION_DATA:
    raise ValueError("A full submission requires Task 2; select both/task2 or a saved two-stage run")
PRIOR_RUN, CLASSIFIER_RUN, INITIAL_ADAPTER, EXPERIMENT_RUN = (
    str(local_path(value, must_exist=True)) if value else None
    for value in (PRIOR_RUN, CLASSIFIER_RUN, INITIAL_ADAPTER, EXPERIMENT_RUN))
if RESUME_CORRECTION:
    INITIAL_ADAPTER = str(resolve_run(PRIOR_RUN, stage="correction") / "adapter")
TRACKING_CONFIG = {"enabled": TRACKING_ENABLED, "project": WANDB_PROJECT,
                   "entity": WANDB_ENTITY, "mode": WANDB_MODE,
                   "log_models": WANDB_LOG_MODELS, "log_text": WANDB_LOG_TEXT}
os.environ["WANDB_RUN_GROUP"] = WANDB_GROUP or OUTPUT_DIR.name
TRACKING_CONFIG_PATH = OUTPUT_ROOT / f"tracking-config-{RUN_ID}.json"
TRACKING_CONFIG_PATH.write_text(json.dumps({"tracking": TRACKING_CONFIG}, indent=2), encoding="utf-8")
command = None
PREFLIGHT_LOG = None
if RUN_TRAINING:
    config = json.loads(local_path(CLASSIFICATION_CONFIG, must_exist=True).read_text(encoding="utf-8"))
    correction_config = json.loads(local_path(CORRECTION_CONFIG, must_exist=True).read_text(encoding="utf-8"))
    config["oof"]["enabled"] = GENERATE_OOF
    apply_overrides(config, CLASSIFICATION_OVERRIDES)
    apply_overrides(correction_config, CORRECTION_OVERRIDES)
    if EDIT_CORRECTION:
        correction_config["inference"]["output_format"] = "edits"
    config["data"].update(path=str(local_path(ORIGINAL_DATA)) if ORIGINAL_DATA else None,
                          augmentation_path=str(local_path(CLASSIFIER_DATA)) if CLASSIFIER_DATA else None,
                          correction_path=str(local_path(CORRECTION_DATA)) if CORRECTION_DATA else None,
                          input_root=str(DATA_ROOT))
    correction_config["data"].update(path=str(local_path(CORRECTION_DATA)) if CORRECTION_DATA else None,
                                    input_root=str(DATA_ROOT))
    for stage in (config, correction_config):
        # Local defaults change placement/paths/tracking only, never the selected recipe.
        stage["run"].update(output_root=str(OUTPUT_ROOT), device=TRAINING_DEVICE, world_size=WORLD_SIZE)
        stage["tracking"] = dict(TRACKING_CONFIG)
    fresh_classifier = RUN_MODE != "task2" and not (CLASSIFIER_RUN or PRIOR_RUN)
    needs_correction = RUN_MODE != "task1" or (fresh_classifier and config["oof"]["enabled"])
    if fresh_classifier:
        original_path = resolve_data(config)
        mixed_config = json.loads(json.dumps(config))
        mixed_config["data"].update(path=config["data"]["augmentation_path"],
                                   filename=config["data"]["augmentation_filename"])
        classifier_path = resolve_data(mixed_config)
        config["data"].update(path=str(original_path), augmentation_path=str(classifier_path))
    if needs_correction:
        correction_path = resolve_data(correction_config)
        config["data"]["correction_path"] = str(correction_path)
        correction_config["data"]["path"] = str(correction_path)
    CONFIG_PATH = OUTPUT_ROOT / f"classification-config-{RUN_ID}.json"
    CORRECTION_CONFIG_PATH = OUTPUT_ROOT / f"correction-config-{RUN_ID}.json"
    for path, stage in ((CONFIG_PATH, config), (CORRECTION_CONFIG_PATH, correction_config)):
        path.write_text(json.dumps(stage, ensure_ascii=False, indent=2), encoding="utf-8")
    load_config(CONFIG_PATH, "classification")
    load_config(CORRECTION_CONFIG_PATH, "correction")
    command = [TRAINING_PYTHON, "-m", "model.two_phase", "--mode", RUN_MODE,
               "--classification-config", str(CONFIG_PATH), "--correction-config", str(CORRECTION_CONFIG_PATH),
               "--output-dir", str(OUTPUT_DIR)]
    if SUBMISSION_DATA:
        command.extend(["--submission-data", SUBMISSION_DATA])
    reuse_flag = "--classifier-run" if CLASSIFIER_RUN else "--prior-run"
    for flag, value in ((reuse_flag, CLASSIFIER_RUN or PRIOR_RUN), ("--initial-adapter", INITIAL_ADAPTER)):
        if value:
            command.extend([flag, value])
    PREFLIGHT_LOG = OUTPUT_ROOT / f"{OUTPUT_DIR.name}-preflight.log"
    stream_command(command + ["--preflight-only"], PREFLIGHT_LOG)
elif not (EXPERIMENT_RUN or PRIOR_RUN):
    raise ValueError("Saved inference/diagnostics require EXPERIMENT_RUN or PRIOR_RUN")


In [ ]:
LOG_PATH = OUTPUT_ROOT / f"{OUTPUT_DIR.name}.log"
if RUN_TRAINING:
    stream_command(command, LOG_PATH)
if not RUN_TRAINING and SUBMISSION_DATA:
    inference_command = [TRAINING_PYTHON, "-m", "model.pipeline_predict",
                         "--run", EXPERIMENT_RUN or PRIOR_RUN, "--data", SUBMISSION_DATA,
                         "--output", str(OUTPUT_DIR / "submission.json"), "--device", DIAGNOSTICS_DEVICE]
    stream_command(inference_command, LOG_PATH)
EXPERIMENT_OUTPUT = None
if HINT_ABLATION or CANDIDATE_RERANKING:
    experiment_run = EXPERIMENT_RUN or (str(OUTPUT_DIR) if RUN_TRAINING else PRIOR_RUN)
    EXPERIMENT_OUTPUT = OUTPUT_ROOT / f"correction-experiments-{RUN_ID}"
    experiment_command = [TRAINING_PYTHON, "-m", "model.correction_experiments",
                          "--run", str(experiment_run), "--output-dir", str(EXPERIMENT_OUTPUT),
                          "--device", DIAGNOSTICS_DEVICE, "--tracking-config", str(TRACKING_CONFIG_PATH)]
    if HINT_ABLATION:
        experiment_command.append("--hint-ablation")
    if CANDIDATE_RERANKING:
        experiment_command.append("--rerank")
    stream_command(experiment_command, EXPERIMENT_OUTPUT.with_suffix(".log"))


In [ ]:
from IPython.display import JSON, FileLink, display

if RUN_TRAINING:
    for filename in ("classification/metrics.json", "correction/metrics.json", "submission.metrics.json", "pipeline_manifest.json", "run_handoff.json"):
        path = OUTPUT_DIR / filename
        if path.is_file():
            print(filename)
            display(JSON(json.loads(path.read_text(encoding="utf-8"))))
    for path in (PREFLIGHT_LOG, LOG_PATH):
        display(FileLink(str(path)))
    print("Saved run:", OUTPUT_DIR)
    print("Reuse it by setting PRIOR_RUN to this local run path; no upload or copy is needed.")
if (OUTPUT_DIR / "submission.json").is_file():
    display(FileLink(str(OUTPUT_DIR / "submission.json")))
    print("Submission scope:", json.loads((OUTPUT_DIR / "submission.evaluation.json").read_text(encoding="utf-8")))
if EXPERIMENT_OUTPUT:
    print("Paired correction diagnostics:", EXPERIMENT_OUTPUT)
    for path in sorted(EXPERIMENT_OUTPUT.rglob("*metrics*.json")):
        print(path.relative_to(EXPERIMENT_OUTPUT))
        display(JSON(json.loads(path.read_text(encoding="utf-8"))))
    display(FileLink(str(EXPERIMENT_OUTPUT.with_suffix(".log"))))
print("Keep local run folders, logs and text-containing diagnostic outputs private.")


## Reuse, warmstart and correction experiments

`RUN_MODE="task1"` trains only the classifier and optional OOF fits. `RUN_MODE="task2"` requires `CLASSIFIER_RUN` or `PRIOR_RUN`. `RUN_MODE="both"` plus `PRIOR_RUN` also reuses the saved classifier and trains Task 2. Saved classifier/tokenizer, OOF hints and provenance are copied into the new portable handoff. Use an explicit extracted run path when a directory contains several runs.

Task 2 starts fresh by default. `RESUME_CORRECTION=True` warmstarts the compatible prior correction adapter using a **new** optimizer and scheduler; it is not exact checkpoint resume. `INITIAL_ADAPTER` chooses an explicit compatible corrector instead. Model revision, LoRA layout, hint mode and sentence/edit representation must agree. Changed data invalidates old OOF coverage: regenerate Task 1 and OOF on the new audited build.

`EDIT_CORRECTION=True` explicitly trains character edits for a fresh adapter; it does not convert an existing sentence adapter. `HINT_ABLATION=True` evaluates paired saved-adapter predicted-hint/no-hint prompts. `CANDIDATE_RERANKING=True` evaluates beam candidates plus unchanged source using frozen-base grammaticality and a CPU multilingual semantic encoder; it adds downloads/inference cost and never guarantees an improvement. Use `RUN_TRAINING=False`, `PRIOR_RUN` or `EXPERIMENT_RUN`, and the experiment toggles to evaluate a saved corrector without fitting again. Diagnostics validate saved holdout IDs, OOF provenance and adapter contracts before generation; their holdout scores are exploratory, not an independent test score.

Interrupting a running job signals its full subprocess group and waits for cleanup, then kills an unresponsive group after 30 seconds. Partial outputs are not a completed reusable run; rerun the parameters cell to create a new output ID rather than overwriting a failed run. W&B offline logs can be synced with the same training environment's `wandb sync` command after the job completes.

For public-test inference with saved adapters, set `RUN_TRAINING=False`, `PRIOR_RUN` and leave `SUBMISSION_DATA="auto"` (or set an explicit input path). Both adapters are required and every row reaches Task 2. From the repository root:

```bash
python -m model.pipeline_predict --run runs/RUN --data dataset/newdataset/public_test.json \
  --output runs/new-submission/submission.json
```

`submission.json` contains only organizer `ID` / `Output` fields. Comparisons and labeled metrics are separate private artifacts. Invalid/blank corrections fail submission export rather than inventing successful answers. Combined scores are a non-official equal-weight mean of macro-F1 and error-only GLEU+; selection remains unchanged.
